# BTW Module 04: Functional & Pathway Enrichment Analysis
**Downstream Bulk Transcriptomics Workbench (`btw`)**

สมุดงานตัวอย่างสาธิตการทำงานของ **FR-5 (Functional & Pathway Enrichment Analysis)** ครบวงจร:
1. **Over-Representation Analysis (ORA):** การคัดกรองยีนและทดสอบความสัมพันธ์ของชีวเส้นทาง (Enrichr / Custom Fisher's Exact Test ผ่าน `goatools`/`scipy`)
2. **Gene Set Enrichment Analysis (GSEA):** การจัดลำดับยีนตามสถิติ (`stat` หรือ `log2FoldChange`) และรัน Fast Prerank ด้วย `gseapy`
3. **Biological Activity Inference (`decoupler`):** การอนุมานระดับการทำงานของ Transcription Factor Regulons (CollecTRI) และ Signaling Pathways (PROGENy) ด้วย Linear Models (ULM / MLM)
4. **Unified Schema & Caching:** โครงสร้างตารางผลลัพธ์มาตรฐาน (`EnrichmentResult`) พร้อมระบบ Caching อัตโนมัติ (`joblib.Memory`)
5. **Publication-Grade Visualizations:** สร้าง Dot Plot และ Bar Plot มาตรฐานสากล ทั้งแบบ Static (Nature/Cell style) และ Interactive (Plotly)
6. **Exporting:** ส่งออกผลการวิเคราะห์ Pathway เป็น Excel หลายชีตและ CSV สำหรับรายงานวิจัย

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import btw
from btw import set_seed, logger
from btw.de_analysis import run_de
from btw.enrichment import (
    EnrichmentResult,
    extract_significant_genes,
    run_custom_ora,
    run_enrichr,
    prepare_ranked_gene_list,
    run_prerank,
    prepare_decoupler_input,
    run_decoupler_activity,
    plot_enrichment_dotplot,
    plot_enrichment_barplot,
    get_memory_cache,
)
from btw.viz import set_publication_style, save_figure
from btw.io import export_table

# กำหนด seed กลางเพื่อความเที่ยงตรงและสไตล์ภาพสำหรับการตีพิมพ์
set_seed(42)
set_publication_style()
print(f"BTW version: {btw.__version__}")

## 1. เตรียมชุดข้อมูลการแสดงออกของยีน (Differential Expression Input)
สร้างตัวอย่างผลลัพธ์ DE analysis จาก PyDESeq2 (100 genes, treated vs control) เพื่อใช้ในการทดสอบกระบวนการทางชีวสารสนเทศ

In [ ]:
# 1. สร้าง Synthetic Counts และ Metadata
np.random.seed(42)
n_genes, n_samples = 100, 6
gene_ids = [f"GENE_{i:03d}" for i in range(1, n_genes + 1)]
sample_ids = [f"Sample_{i:02d}" for i in range(1, n_samples + 1)]

counts = np.random.negative_binomial(n=20, p=0.05, size=(n_genes, n_samples))
# กำหนด 15 ยีนแรกเป็น UP-regulated ใน treated
counts[:15, 3:] = (counts[:15, 3:] * 4.5).astype(int)
# กำหนด 15 ยีนถัดมาเป็น DOWN-regulated ใน treated
counts[15:30, :3] = (counts[15:30, :3] * 4.0).astype(int)

counts_df = pd.DataFrame(counts, index=gene_ids, columns=sample_ids)
meta_df = pd.DataFrame({
    "condition": ["control"] * 3 + ["treated"] * 3,
    "batch": ["b1", "b2", "b1", "b2", "b1", "b2"],
}, index=sample_ids)

# 2. รัน Differential Expression ด้วย BTW Thin Helper
de_res = run_de(
    counts=counts_df,
    metadata=meta_df,
    contrast=("condition", "treated", "control"),
    alpha=0.05,
    lfc_threshold=1.0,
)
print(de_res.summary())

## 2. Over-Representation Analysis (ORA)
ทดสอบความชุกของยีนกลุ่มที่มีนัยสำคัญ (UP / DOWN) ในชีวเส้นทางต่าง ๆ ด้วย Fisher's Exact Test และสืบค้นผ่านฐานข้อมูลชีววิทยา

In [ ]:
# 1. คัดกรองยีนกลุ่ม UP-regulated และ DOWN-regulated
up_genes = extract_significant_genes(de_res, direction="up")
down_genes = extract_significant_genes(de_res, direction="down")
print(f"Significant UP genes ({len(up_genes)}): {up_genes[:5]}...")
print(f"Significant DOWN genes ({len(down_genes)}): {down_genes[:5]}...")

# 2. กำหนด Gene Sets ตัวอย่างสำหรับ Pathway Testing
gene_sets = {
    "Cell_Cycle_G1_S": ["GENE_001", "GENE_002", "GENE_003", "GENE_004", "GENE_005"],
    "DNA_Repair_Pathway": ["GENE_006", "GENE_007", "GENE_008", "GENE_009", "GENE_010"],
    "Oxidative_Phosphorylation": ["GENE_016", "GENE_017", "GENE_018", "GENE_019", "GENE_020"],
    "Lipid_Metabolism": ["GENE_080", "GENE_081", "GENE_082", "GENE_083", "GENE_084"],
}

# 3. รัน ORA ผ่าน Fisher's Exact Test
ora_res = run_custom_ora(
    gene_list=up_genes,
    gene_sets=gene_sets,
    background=gene_ids,
    alpha=0.05,
)
print(ora_res.summary())
print(ora_res.results_df.head())

## 3. Gene Set Enrichment Analysis (GSEA)
วิเคราะห์การกระจายตัวของยีนทั้งชุดข้อมูลโดยไม่อาศัยการตัด cutoff ของ p-value โดยคำนวณ Normalized Enrichment Score (NES) และ FDR q-value ผ่าน `gseapy.prerank`

In [ ]:
# 1. จัดเตรียม Ranked Gene Series ตาม Wald Statistic
ranked_genes = prepare_ranked_gene_list(de_res, rank_by="stat")
print(f"Top ranked genes:\n{ranked_genes.head(3)}\n")
print(f"Bottom ranked genes:\n{ranked_genes.tail(3)}\n")

# 2. รัน GSEA Prerank ด้วย fast permutation
gsea_res = run_prerank(
    ranked_genes=ranked_genes,
    gene_sets=gene_sets,
    min_size=2,
    permutation_num=50,
    seed=42,
    alpha=0.05,
)
print(gsea_res.summary())
gsea_res.results_df

## 4. Biological Activity Inference (`decoupler`)
อนุมานระดับการทำงานของ Transcription Factors (Regulons) หรือ Signaling Pathways โดยอาศัยข้อมูล Prior Knowledge Network ร่วมกับสถิติของยีนจาก Differential Expression

In [ ]:
# 1. สร้าง Prior Knowledge Network ตัวอย่าง (Regulon-Target Interactions พร้อม Weights)
net_df = pd.DataFrame({
    "source": ["E2F_TF", "E2F_TF", "E2F_TF", "MYC_TF", "MYC_TF", "TP53_TF", "TP53_TF"],
    "target": ["GENE_001", "GENE_002", "GENE_003", "GENE_004", "GENE_005", "GENE_016", "GENE_017"],
    "weight": [1.2, 1.0, 0.8, 1.1, 1.3, -1.0, -0.9],
})

# 2. แปลงผลลัพธ์ DE เป็น Matrix สำหรับ decoupler
mat_df = prepare_decoupler_input(de_res, metric="stat")

# 3. คำนวณ Activity Scores ด้วย Univariate Linear Model (ULM)
act_res = run_decoupler_activity(
    mat=mat_df,
    net=net_df,
    method="ulm",
    min_n=2,
    alpha=0.05,
    network_name="Custom_TF_Regulons",
)
print(act_res.summary())
act_res.results_df

## 5. Publication-Grade Visualizations
สร้างภาพประกอบผลการวิเคราะห์ Pathway ระดับวารสารวิชาการชั้นนำ:
- **Dot Plot:** ขนาดจุดแสดงจำนวนยีน (Gene Count), สีแสดงความมีนัยสำคัญ (-log10 padj), ตำแหน่งแกน X แสดง Enrichment Score / NES / Activity
- **Bar Plot:** ความยาวแท่งแสดง Effect Size / Score, สีแสดงระดับนัยสำคัญ

In [ ]:
fig_dir = Path("reports/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

# 1. Unified Dot Plot (Static)
fig_dot, ax_dot = plot_enrichment_dotplot(
    enrichment=ora_res,
    top_n=10,
    title="ORA Pathway Enrichment Dot Plot (UP Genes)",
)
save_figure(fig_dot, fig_dir / "04_ora_dotplot.png", dpi=300)
plt.close(fig_dot)

# 2. GSEA Bar Plot (Static)
fig_bar, ax_bar = plot_enrichment_barplot(
    enrichment=gsea_res,
    metric="score",
    title="GSEA Normalized Enrichment Scores (NES)",
)
save_figure(fig_bar, fig_dir / "04_gsea_barplot.png", dpi=300)
plt.close(fig_bar)

# 3. Interactive Dot Plot ด้วย Plotly
plotly_fig = plot_enrichment_dotplot(
    enrichment=act_res,
    interactive=True,
    title="Transcription Factor Regulon Activities (ULM)",
)
# In Jupyter, evaluating plotly_fig displays interactive widget
print(f"Created interactive Plotly figure with {len(plotly_fig.data)} traces.")

## 6. Result Caching & Multi-Sheet Excel Export
บันทึกผลลัพธ์ที่คำนวณผ่าน Caching (`joblib.Memory`) และส่งออกผลการวิเคราะห์ ORA, GSEA, และ TF Activity สู่ Excel หลายชีต

In [ ]:
# 1. ทดสอบ Caching wrapper
cache = get_memory_cache(cache_dir=".cache_enrichment", enabled=True)

@cache.cache
def cached_analysis(genes):
    logger.info("Executing expensive pathway analysis...")
    return len(genes)

val1 = cached_analysis(tuple(up_genes))
val2 = cached_analysis(tuple(up_genes))  # เรียกจาก cache โดยตรง
assert val1 == val2

# 2. รวบรวมตารางผลลัพธ์มาตรฐานทั้งหมดส่งออกเป็น Multi-sheet Excel
out_excel = Path("reports/enrichment_results.xlsx")
export_table(
    ora_res.results_df,
    output_path=out_excel,
    sheet_name="ORA_UP",
    index=False,
)
with pd.ExcelWriter(out_excel, mode="a", engine="openpyxl") as writer:
    gsea_res.results_df.to_excel(writer, sheet_name="GSEA_Prerank", index=False)
    act_res.results_df.to_excel(writer, sheet_name="TF_Activity_ULM", index=False)

print(f"Successfully exported multi-sheet enrichment results to: {out_excel}")
print("Module 04 execution completed successfully!")